# 1 · Opening the dataset

This notebook shows how to load every part of the **Volve ML-ready** dataset and how its
indices and coordinates fit together. You only need `numpy` and `pandas`. The
`volve_mlready` helper is a thin convenience layer, and each step also shows the plain
`numpy`/`pandas` call.

**Before you start:** download the files from Zenodo (DOI `10.5281/zenodo.TODO`) into
`data/volve-mlready-v1.0.0/`, or set `VOLVE_MLREADY_DIR`. See the README for the
`volve_mlready.download()` helper.

In [1]:
import os
import numpy as np
import pandas as pd
from volve_mlready import VolveMLReady

# folder holding the files downloaded from Zenodo
DATA = os.environ.get('VOLVE_MLREADY_DIR', '../data/volve-mlready-v1.0.0')
v = VolveMLReady(DATA)

## What is in the folder

In [2]:
files = sorted(p for p in v.root.iterdir() if p.is_file())
pd.DataFrame({'file': [p.name for p in files],
              'MB': [round(p.stat().st_size / 1e6, 1) for p in files]})

,file,MB
0,CHANGELOG.md,0.0
1,CITATION.cff,0.0
2,DATA_LICENSE.md,0.0
3,Equinor_Open_Data_Licence_Volve.pdf,0.3
4,README.md,0.0
5,SHA256SUMS.txt,0.0
6,code.zip,0.0
7,grid_st0202_depth.json,0.0
8,pp_amplitude_on_st0202_depth.npy,839.5
9,pp_poststack_st0202_time.json,0.0


Optional integrity check against `SHA256SUMS.txt` (reads all 3.2 GB, takes a few seconds
to a minute):

In [3]:
from volve_mlready import verify
status = verify(DATA)
print({k: n for k, n in pd.Series(status).value_counts().items()})

{'ok': 16}


## The common grid

The three main cubes share one grid, described in `grid_st0202_depth.json`. Arrays are
indexed `C[iz, iy, ix]`: depth below mean sea level, inline, crossline.

In [4]:
g = v.grid
print('shape (z, inline, crossline):', g['shape_zyx'])
print(f"depth : z = {g['z0']} + {g['dz']}*iz   ->  {g['depths'][0]}-{g['depths'][-1]} m below MSL")
print(f"inline: IL = {g['il0']} + iy           ->  {g['ilines'][0]}-{g['ilines'][-1]}")
print(f"xline : XL = {g['xl0']} + ix           ->  {g['xlines'][0]}-{g['xlines'][-1]}")
print('CRS   :', g['crs'])
print('wave  :', g['wave_mode'])

shape (z, inline, crossline): [901, 385, 605]
depth : z = 0.0 + 5.0*iz   ->  0.0-4500.0 m below MSL
inline: IL = 9985 + iy           ->  9985-10369
xline : XL = 1932 + ix           ->  1932-2536
CRS   : ED50 / UTM zone 31N (EPSG:23031)
wave  : PS (converted wave), PSDM full offset, post-stack


## The cubes

`np.load(..., mmap_mode='r')` maps the file without reading it, so slicing is cheap and the
840 MB arrays never have to fit in memory at once.

In [5]:
cubes = {name: v.cube(name) for name in ['ps_amplitude', 'vp', 'pp_amplitude']}
# the same without the helper:
amp_plain = np.load(v.root / 'st0202_ps_amplitude_depth.npy', mmap_mode='r')

pd.DataFrame({name: {'shape': c.shape, 'dtype': str(c.dtype), 'GB': round(c.nbytes / 1e9, 2)}
              for name, c in cubes.items()}).T

,shape,dtype,GB
ps_amplitude,"(901, 385, 605)",float32,0.84
vp,"(901, 385, 605)",float32,0.84
pp_amplitude,"(901, 385, 605)",float32,0.84


In [6]:
# summary statistics on a 1-in-5 subsample of the reservoir interval (2500-3400 m)
iz0, iz1 = int(2500 / g['dz']), int(3400 / g['dz'])
rows = {}
for name, c in cubes.items():
    s = np.asarray(c[iz0:iz1:5, ::5, ::5], dtype=np.float64)
    rows[name] = dict(min=s.min(), p1=np.percentile(s, 1), median=np.median(s),
                      p99=np.percentile(s, 99), max=s.max())
pd.DataFrame(rows).T.map(lambda x: float(f'{x:.4g}'))

,min,p1,median,p99,max
ps_amplitude,-0.26560,-0.07324,0.000171,0.07104,0.20580
vp,2698.00000,2856.00000,3895.000000,5042.00000,5319.00000
pp_amplitude,-0.03651,-0.01876,0.000110,0.02366,0.04715


## Coordinates

`(inline, crossline) → (X, Y)` is an affine transform in ED50 / UTM 31N. The helper exposes
it both ways, and also between grid indices and world coordinates.

In [7]:
X, Y, z = v.index_to_world(iz=590, iy=200, ix=300)
print(f'cell (590, 200, 300) centre: X={float(X):.1f} m  Y={float(Y):.1f} m  z={float(z):.0f} m TVDSS')
print('back to indices:', v.world_to_index(X, Y, z))

cell (590, 200, 300) centre: X=435693.2 m  Y=6478847.4 m  z=2950 m TVDSS
back to indices: (np.int64(590), np.int64(200), np.int64(300))


## The wells

Three tables live inside `wells.zip`, plus one trajectory per well:

* `index`: one row per wellbore (sources, datum, counts);
* `voxels`: the **training table**, one row per (well, voxel), with the target `PHIE`;
* `samples`: the original log samples (0.1–0.15 m), before upscaling.

In [8]:
idx = v.wells('index')
idx[['npd', 'wellbore', 'survey_file', 'EDF_m', 'max_inclination_deg',
     'n_samples', 'n_voxels', 'TVDSS_min', 'TVDSS_max', 'PHIE_mean']]

,npd,wellbore,survey_file,EDF_m,max_inclination_deg,n_samples,n_voxels,TVDSS_min,TVDSS_max,PHIE_mean
0,NPD-3145,15/9-19 A,NPD-3145.csv,25.0,61.37,1967,68,2873.81,3143.36,0.1404
1,NPD-7405,15/9-F-1C,NPD-7405.csv,54.9,84.06,8422,136,2883.14,3116.08,0.1768
2,NPD-7079,15/9-F-11A,NPD-7079.csv,54.9,75.37,1533,36,2928.28,3052.20,0.1595
3,NPD-7080,15/9-F-11B,NPD-7080.csv,54.9,90.01,13933,207,2791.74,3182.26,0.1547
4,NPD-5599,15/9-F-12,NPD-5599.csv,54.9,54.95,2647,77,2804.32,3044.62,0.1743
5,NPD-2105,15/9-19 SR,NPD-2105.csv,25.0,61.37,6886,217,2370.26,3090.93,0.1083
6,NPD-7223,15/9-F-1,NPD-7223.csv,54.9,49.50,2872,65,2990.71,3251.96,0.0823
7,NPD-7224,15/9-F-1A,NPD-7224.csv,54.9,63.21,2281,46,3003.21,3167.13,0.0901
8,NPD-7264,15/9-F-1B,NPD-7264.csv,54.9,30.25,2105,47,2971.46,3179.86,0.1168
9,NPD-7289,15/9-F-15D,NPD-7289.csv,54.9,82.04,11936,160,2852.60,3152.84,0.1661


In [9]:
vox = v.wells('voxels')
smp = v.wells('samples')
print(f'{len(vox):,} voxels and {len(smp):,} samples from {vox.npd.nunique()} wellbores')
vox.head()

1,105 voxels and 56,599 samples from 11 wellbores


,well,npd,wellbore,IX,IY,IZ,PHIE,PHIE_std,n_samples,MD,TVD,TVDSS,EDF,X,Y,IL,XL
0,0,NPD-3145,15/9-19 A,257,203,624,0.082927,0.027259,18,3941.5974,3146.200891,3121.200891,25.0,436218.764069,6.478758e+06,10188.266572,2189.484789
1,0,NPD-3145,15/9-19 A,257,203,625,0.102626,0.037868,37,3945.7884,3149.974143,3124.974143,25.0,436219.769157,6.478760e+06,10188.404173,2189.436229
2,0,NPD-3145,15/9-19 A,257,203,626,0.169211,0.000000,1,3948.6840,3152.581117,3127.581117,25.0,436220.463581,6.478761e+06,10188.499244,2189.402678
3,0,NPD-3145,15/9-19 A,257,204,626,0.159274,0.033094,35,3951.4272,3155.050882,3130.050882,25.0,436221.121456,6.478762e+06,10188.589311,2189.370893
4,0,NPD-3145,15/9-19 A,257,204,627,0.168865,0.029108,37,3956.9136,3159.990413,3134.990413,25.0,436222.437207,6.478764e+06,10188.769444,2189.307323


In [10]:
# the same without the helper
import zipfile
with zipfile.ZipFile(v.root / 'wells.zip') as z:
    print(z.namelist()[:5], '...')
    vox_plain = pd.read_csv(z.open('wells_voxels.csv'))
assert vox_plain.equals(vox)

['trajectories/F-11T2.csv', 'trajectories/NPD-2105.csv', 'trajectories/NPD-3145.csv', 'trajectories/NPD-5599.csv', 'trajectories/NPD-7079.csv'] ...


In [11]:
v.trajectory('NPD-7080').iloc[::40]       # 15/9-F-11 B, a horizontal producer

,MD,INC,AZI,TVD_DF,TVDSS,X,Y
0,145.9,0.00,0.00,145.90,91.00,435049.095,6478568.172
40,540.0,2.13,310.13,539.88,484.98,435041.867,6478572.026
80,940.0,10.58,264.05,936.14,881.24,435000.718,6478559.997
120,1340.0,21.31,283.42,1317.74,1262.84,434890.510,6478586.125
160,1740.0,32.35,258.27,1660.30,1605.40,434689.732,6478558.191
200,2140.0,2.93,341.18,2039.32,1984.42,434580.772,6478546.196
240,2540.0,34.06,67.96,2415.95,2361.05,434685.720,6478596.435
280,4085.0,76.09,97.76,2966.87,2911.97,435975.503,6478298.588


## Cube values at the wells

`IZ, IY, IX` in the voxel table index straight into any of the three cubes, so building a
feature table is one line per cube.

In [12]:
feat = pd.DataFrame({name: np.asarray(c[vox.IZ.values, vox.IY.values, vox.IX.values])
                     for name, c in cubes.items()})
feat['PHIE'] = vox.PHIE.values
feat['well'] = vox.wellbore.values
feat.head()

,ps_amplitude,vp,pp_amplitude,PHIE,well
0,-0.017496,3697.165771,-0.004283,0.082927,15/9-19 A
1,-0.009644,3704.742188,-0.003452,0.102626,15/9-19 A
2,-0.001167,3712.318848,-0.002349,0.169211,15/9-19 A
3,-0.004220,3707.886719,-0.002513,0.159274,15/9-19 A
4,-0.001515,3715.457031,-0.001399,0.168865,15/9-19 A


For context-based models, cut a window around each voxel. Here is a vertical trace of
±10 samples (±50 m), with edge padding:

In [13]:
def vertical_windows(cube, iz, iy, ix, half=10):
    nz = cube.shape[0]
    offs = np.arange(-half, half + 1)
    zz = np.clip(iz[:, None] + offs[None, :], 0, nz - 1)
    return np.asarray(cube[zz, iy[:, None], ix[:, None]])

W = vertical_windows(cubes['ps_amplitude'], vox.IZ.values, vox.IY.values, vox.IX.values)
print('windows:', W.shape, '(voxels, samples)')

windows: (1105, 21) (voxels, samples)


## Validation: group by well

Neighbouring voxels of the same well are strongly correlated, and some wells share a slot
(the F-1 and F-11 families). **Split by well, never by voxel.** For example, use
leave-one-well-out:

In [14]:
groups = vox.npd.values
for test_well in np.unique(groups)[:3]:
    train, test = groups != test_well, groups == test_well
    print(f'{test_well}: train {train.sum():4d} voxels / test {test.sum():3d} voxels')
print('...')

F-11T2: train 1059 voxels / test  46 voxels
NPD-2105: train  888 voxels / test 217 voxels
NPD-3145: train 1037 voxels / test  68 voxels
...


Next: [`02_exploration.ipynb`](02_exploration.ipynb) looks at the data.

*Contains data from the Volve field dataset, provided by Equinor ASA and the former Volve
licence partners ExxonMobil Exploration & Production Norway AS and Bayerngas Norge AS, under
the Equinor Open Data Licence.*